In [1]:
import sys
# Link your local downloads path so the notebook can see your custom script file
sys.path.append(r"C:\Users\91880\Downloads")

import numpy as np
from qi_qiskit_sensing import sensing_pd_pfa

# --- 1. Spectrum Sensing Configuration & Caching ---
# Dictionary to pre-calculate and cache Qiskit results for speed
SENSING_CACHE = {}

def get_sensing_metrics(snr_db, strategy="quantum"):
    """Fetches cached simulated (Pd, Pfa) or runs the Qiskit circuit if new."""
    snr_key = (round(float(snr_db), 1), strategy)
    if snr_key not in SENSING_CACHE:
        SENSING_CACHE[snr_key] = sensing_pd_pfa(snr_db, strategy=strategy)
    return SENSING_CACHE[snr_key]

# --- 2. Defining Users, Channels, and SINR Simulation ---
n_users = 50      # secondary users
n_channels = 60   # available spectrum bands

np.random.seed(67)  # Using seed from your QPSO script to maintain consistency
channel_gain = np.random.uniform(0.1, 1.0, (n_users, n_channels))
noise = 0.1
SINR = channel_gain / noise

# Set up active primary user channels
n = n_channels          
k = 7           
arr = np.zeros(n, dtype=int)
arr[np.random.choice(n, k, replace=False)] = 1
PU_occupied = arr

# --- 3. Modified Fitness Function incorporating Quantum Illumination ---
def fitness_with_sensing(x, strategy="quantum"):
    channels = np.clip(np.round(x).astype(int), 0, n_channels - 1)
    throughput = 0
    penalty = 0
    
    for user in range(n_users):
        ch = channels[user]
        snr_linear = SINR[user, ch]
        snr_db = 10 * np.log10(snr_linear)
        
        # Get physical Pd and Pfa from the Qiskit simulation script
        Pd, Pfa = get_sensing_metrics(snr_db, strategy=strategy)
        
        if PU_occupied[ch]:
            # Hard interference penalty weighted by miss-rate (1 - Pd)
            penalty += (1.0 - Pd) * 1000
        else:
            # Successful transmission rate weighted by false alarm constraint (1 - Pfa)
            throughput += (1.0 - Pfa) * np.log2(1 + SINR[user, ch])
            
    # Secondary user collision penalty (co-channel collision rule)
    for ch in range(n_channels):
        users_on_ch = np.sum(channels == ch)
        if users_on_ch > 1:
            penalty += (users_on_ch - 1) * 1000
            
    # Return negative since QPSO minimizes the objective function
    return -throughput + penalty

# --- 4. QPSO Algorithm Core Implementation ---
def qpso(f, N, beta_start, beta_end, n_iter, D, low, high):
    # Initialize positions
    x = np.random.uniform(low, high, size=(N, D))
    
    # Personal bests
    p = x.copy()
    p_fitness = np.array([f(p[i]) for i in range(N)])
    
    # Global best
    g = p[np.argmin(p_fitness)].copy()
    
    for t in range(n_iter):
        # Progress logging
        if t % 100 == 0:
            print(f"Iteration {t}/{n_iter}...")
            
        # Beta decreases linearly from beta_start to beta_end
        beta = beta_start - (beta_start - beta_end) * t / n_iter
        
        # Mean best position — average of all personal bests
        mbest = np.mean(p, axis=0)
        
        for i in range(N):
            # Local attractor — random point between personal best and global best
            phi = np.random.uniform(0, 1, D)
            p_local = phi * p[i] + (1 - phi) * g
            
            # Sample new position from Laplace distribution
            u = np.random.uniform(0, 1, D)
            sign = np.where(np.random.rand(D) < 0.5, 1, -1)
            x[i] = p_local + sign * beta * np.abs(mbest - x[i]) * np.log(1/u)
            
            # Clip to valid range
            x[i] = np.clip(x[i], low, high)
            
            # Evaluate objective function
            fx_i = f(x[i])
            
            # Update personal best
            if fx_i < p_fitness[i]:
                p[i] = x[i].copy()
                p_fitness[i] = fx_i
                
                # Update global best
                if p_fitness[i] < f(g):
                    g = p[i].copy()
                    
    return g

# --- 5. Run QPSO with Quantum Illumination Sensing ---
# Wrapping objective function to use "quantum" strategy strategy
quantum_objective = lambda x: fitness_with_sensing(x, strategy="quantum")

result_qpso = qpso(
    f=quantum_objective,
    N=150,
    beta_start=1.0,
    beta_end=0.5,
    n_iter=1000,
    D=n_users,
    low=0,
    high=n_channels-1
)

# --- 6. Post-Processing & Metrics Reporting ---
best_assignment = np.clip(np.round(result_qpso).astype(int), 0, n_channels-1)
actual_throughput = 0

for user in range(n_users):
    ch = best_assignment[user]
    if not PU_occupied[ch]:
        actual_throughput += np.log2(1 + SINR[user, ch])

print("\nOptimal Channel Assignment:\n", best_assignment)
print("Net Achieved Throughput Under Optimal Allocation:", actual_throughput)

Iteration 0/1000...
Iteration 100/1000...
Iteration 200/1000...
Iteration 300/1000...
Iteration 400/1000...
Iteration 500/1000...
Iteration 600/1000...
Iteration 700/1000...
Iteration 800/1000...
Iteration 900/1000...

Optimal Channel Assignment:
 [32  2 56 58  1 27 52 29 50 47 38 39 13  7 49 33 25 31 21 35 46 43 27 23
 21  0 48 24 42 37 32 36 59 22 40 49 17 59 41 53 18  3  8 30 47 51 19 20
 11 16]
Net Achieved Throughput Under Optimal Allocation: 106.95439119958992
